# 1. BETO jerárquico: pertinencia + seis funciones históricas

Un BETO binario A y un BETO histórico B **independientes desde el mismo modelo base**, por cada semilla
42, 123 y 2026. Combinación soft gating, sin umbrales ajustados ni selección de semilla.
Carga manual: `train-experimental.jsonl` (758) y `dev.jsonl` (160). No contiene datos.
No modifica los textos ni etiquetas originales; las etiquetas de cada tarea se derivan en memoria.
V y S no se cargan. DEV selecciona épocas y ya está expuesto: resultado de desarrollo, no test independiente ni gold experto.

**Uso:** GPU → Ejecutar todo con `RUN_SMOKE_TEST = True` (celda 17). Después cambia esa variable a
`False` y vuelve a ejecutar desde la celda 17 hasta el final para el experimento completo.
El smoke test no produce resultados científicos y sus pesos se descartan. La ejecución completa son seis
entrenamientos: A y B para cada semilla. No se entrena BETO plano aquí.

Se guardan resultados incrementalmente en `/content/results/` y mejores checkpoints en
`/content/checkpoints/`. Una desconexión de Colab puede borrar almacenamiento temporal: descarga los ZIP
o copia estas carpetas a tu Drive antes de cerrar. No hay reanudación automática ni media de semillas incompletas.
El padding dinámico solicitado difiere del padding fijo del notebook plano; registra esa diferencia al comparar.


In [ ]:
# 2. Instalación de dependencias (antes de importar transformers)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'huggingface-hub==0.36.2', 'tokenizers==0.22.2',
    'safetensors==0.8.0', 'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'tqdm'], check=True)
# BERT es texto: evitar conflictos de extensiones opcionales de audio/visión de Colab.
import transformers.utils.import_utils as hf_imports
hf_imports._torchvision_available = False
hf_imports._librosa_available = False


In [ ]:
# 3. Imports
import os, gc, json, math, random, hashlib, time, shutil, platform, zipfile
from pathlib import Path
from collections import Counter
from itertools import islice
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
from google.colab import files
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, set_seed
from huggingface_hub import snapshot_download
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

def save_json(path, value):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+'\n',encoding='utf-8')
    temp.replace(path)

def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()


In [ ]:
# 4. CONFIG — mismo protocolo para todas las semillas y ambos modelos
MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-cased'
MODEL_REVISION = 'c4d86612f51b4f46759c8390d1798c2febe71b93'
MAX_LENGTH = 384
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_EPOCHS = 8
PATIENCE = 2
MIN_DELTA = 1e-4
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0
MICRO_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
SEEDS = [42, 123, 2026]
DROPOUT = 0.1
# Permitir recuperación normal antes de declarar persistencia: 20 omisiones consecutivas.
MAX_CONSECUTIVE_AMP_SKIPS = 20
RESULTS = Path('/content/results')
CHECKPOINTS = Path('/content/checkpoints')
BASE_HASHES = {
 'config.json':'3694a761b0ba882c24baab95df01cf7e7e1424797af557272e944ec2452f9f31',
 'pytorch_model.bin':'e131a95091c777bbd45250fb647fec415010cb8cd1ad6e1d59babeb82a0be360',
 'special_tokens_map.json':'bd6ed009009f8264d0ef87d5b50798cb57c5219a0bbb7c8973372855241aa05f',
 'tokenizer.json':'ea7a58026720ba45a8a401d9f86bbe8337f2725d3b02d3a1011305c46cbbd9cd',
 'tokenizer_config.json':'2f5cde6bbb9959fccd280e9e9c08d41d05e96743fac1a3b5b67e0e5691ae7d10',
 'vocab.txt':'b8f1c939e21273bd19cd885d0b6d7eb11244240ef81f62e30dcf84b4c970ce36'}


In [ ]:
# 5. GPU y reproducibilidad
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['TOKENIZERS_PARALLELISM']='false'
if not torch.cuda.is_available():raise RuntimeError('Activa GPU en Entorno de ejecución > Cambiar tipo.')
if tuple(int(v) for v in torch.__version__.split('+')[0].split('.')[:2]) < (2,6):
    raise RuntimeError('Se requiere PyTorch >=2.6 para cargar el BETO base .bin; usa un runtime Colab actualizado.')
DEVICE=torch.device('cuda')
print('GPU:',torch.cuda.get_device_name(),'PyTorch:',torch.__version__,'CUDA:',torch.version.cuda)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
torch.use_deterministic_algorithms(True)

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed);set_seed(seed)

def release_gpu():
    gc.collect();torch.cuda.empty_cache()

seed_all(SEEDS[0])
print('Semillas fijas; se registra el entorno. GPU/versiones distintas pueden producir diferencias numéricas.')


In [ ]:
# 6. Carga manual TRAIN/DEV. Selecciona ambos archivos en el cuadro de carga.
uploaded = files.upload()
def detect_upload(wanted):
    matches=[name for name in uploaded if Path(name).name == wanted]
    if len(matches)!=1:raise ValueError(f'Carga exactamente un archivo llamado {wanted}; recibidos: {list(uploaded)}')
    return matches[0]
TRAIN_FILE=detect_upload('train-experimental.jsonl')
DEV_FILE=detect_upload('dev.jsonl')
def read_jsonl(name):
    records=[]
    for line_number,line in enumerate(uploaded[name].decode('utf-8-sig').splitlines(),1):
        if not line.strip():continue
        try:record=json.loads(line)
        except Exception as exc:raise ValueError(f'{name}, línea {line_number}: JSON inválido') from exc
        if not isinstance(record,dict):raise ValueError(f'{name}, línea {line_number}: se espera objeto JSON')
        records.append(record)
    return records
train_rows=read_jsonl(TRAIN_FILE)
dev_rows=read_jsonl(DEV_FILE)
input_hashes={'train':hashlib.sha256(uploaded[TRAIN_FILE]).hexdigest(),
              'dev':hashlib.sha256(uploaded[DEV_FILE]).hexdigest()}
# Una carga opcional en este mismo cuadro también se reconoce más adelante.
flat_uploaded=uploaded.get('flat_beto_results.json')
print('TRAIN:',len(train_rows),'DEV:',len(dev_rows))


In [ ]:
# 7. Validaciones críticas. Los metadatos se conservan; nunca son features.
EXPECTED_LABELS=(
 'campanias_conflictos_militares','contexto_colonial_antecedentes','crisis_ideas_emancipadoras',
 'liderazgos_diplomacia_proyectos','no_relevante','organizacion_consecuencias_republicanas','participacion_social_regional')
def text_hash(r):return hashlib.sha256(r['text'].encode('utf-8')).hexdigest()
def validate_split(rows,name,expected):
    if len(rows)!=expected:raise ValueError(f'{name}: esperado {expected}, recibido {len(rows)}')
    for i,r in enumerate(rows):
        for key in ('id','text','label'):
            if not isinstance(r.get(key),str) or not r[key].strip():raise ValueError(f'{name}[{i}]: falta {key} no vacío de tipo string')
        if r['label'] not in EXPECTED_LABELS:raise ValueError(f'{name}: label desconocido {r["label"]!r}')
        if r.get('text_sha256') is not None and r['text_sha256']!=text_hash(r):raise ValueError(f'{name}: hash de texto incorrecto {r["id"]}')
    if len({r['id'] for r in rows})!=len(rows):raise ValueError(f'{name}: IDs duplicados')
    if len({text_hash(r) for r in rows})!=len(rows):raise ValueError(f'{name}: textos exactamente duplicados')
    if set(r['label'] for r in rows)!=set(EXPECTED_LABELS):raise ValueError(f'{name}: faltan clases esperadas')
validate_split(train_rows,'TRAIN',758);validate_split(dev_rows,'DEV',160)
for key,fn in [('id',lambda r:r['id']),('text_hash',text_hash)]:
    if {fn(r) for r in train_rows}&{fn(r) for r in dev_rows}:raise ValueError(f'Solapamiento TRAIN/DEV: {key}')
for key in ('family','component','source_id'):
    a={str(r[key]) for r in train_rows if r.get(key) is not None and str(r[key]).strip()}
    b={str(r[key]) for r in dev_rows if r.get(key) is not None and str(r[key]).strip()}
    if a&b:raise ValueError(f'Solapamiento TRAIN/DEV en {key}: {sorted(a&b)}')
    if not all(r.get(key) for r in train_rows+dev_rows):print(f'{key}: comprobación parcial; metadatos ausentes en algunas filas.')
print('Validaciones críticas correctas. No se creó ni modificó ninguna partición.')


In [ ]:
# 8. Distribución de clases
distribution=pd.DataFrame({'TRAIN':Counter(r['label'] for r in train_rows),'DEV':Counter(r['label'] for r in dev_rows)}).reindex(EXPECTED_LABELS)
display(distribution)


In [ ]:
# 9. Mappings explícitos e inmutables
FINAL_LABELS=list(EXPECTED_LABELS)
A_LABELS=['no_relevante','relevante']
B_LABELS=['contexto_colonial_antecedentes','crisis_ideas_emancipadoras','campanias_conflictos_militares',
          'liderazgos_diplomacia_proyectos','organizacion_consecuencias_republicanas','participacion_social_regional']
FINAL_TO_ID={l:i for i,l in enumerate(FINAL_LABELS)}
B_TO_ID={l:i for i,l in enumerate(B_LABELS)}
NR_ID=FINAL_TO_ID['no_relevante']
assert NR_ID==4 and set(B_LABELS)==set(FINAL_LABELS)-{'no_relevante'}
assert [FINAL_TO_ID[l] for l in B_LABELS]==[1,2,0,3,5,6]
print('Orden final:',FINAL_TO_ID)


In [ ]:
# 10. Tokenización única, padding dinámico, sin alterar texto
BASE_PATH=Path(snapshot_download(MODEL_NAME,revision=MODEL_REVISION,allow_patterns=list(BASE_HASHES)))
for name,expected in BASE_HASHES.items():
    if digest(BASE_PATH/name)!=expected:raise ValueError(f'BETO base/tokenizer cambiado: {name}')
tokenizer=AutoTokenizer.from_pretrained(BASE_PATH,use_fast=True,local_files_only=True)
collator=DataCollatorWithPadding(tokenizer,padding=True,return_tensors='pt')
def tokenize_rows(rows):
    enc=tokenizer([r['text'] for r in rows],truncation=True,max_length=MAX_LENGTH,padding=False)
    return [{k:enc[k][i] for k in enc} for i in range(len(rows))]
train_tokens=tokenize_rows(train_rows);dev_tokens=tokenize_rows(dev_rows)
for name,tokens in [('TRAIN',train_tokens),('DEV',dev_tokens)]:
    print(name,'textos con longitud tokenizada ==384:',sum(len(t['input_ids'])==MAX_LENGTH for t in tokens))
print('Alcanzar 384 no distingue por sí solo longitud exacta de truncamiento.')
def make_loader(rows,tokens,task,shuffle,seed,indices=None):
    if indices is None:indices=range(len(rows))
    items=[]
    for i in indices:
        # Inferencia B sobre DEV completo: etiquetas NR no intervienen en el forward.
        y=int(rows[i]['label']!='no_relevante') if task=='A' else B_TO_ID.get(rows[i]['label'],-100)
        items.append({**tokens[i],'labels':y})
    return torch.utils.data.DataLoader(items,batch_size=MICRO_BATCH_SIZE,shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed),num_workers=0,collate_fn=collator)


In [ ]:
# 11. Métricas y evaluación en FP32
def metrics(y,probabilities,labels):
    p=np.asarray(probabilities,dtype=np.float64);y=np.asarray(y,dtype=int)
    assert p.shape==(len(y),len(labels)) and np.isfinite(p).all()
    assert (p>=0).all() and np.allclose(p.sum(1),1,atol=1e-6)
    pred=p.argmax(1)
    precision,recall,f1,support=precision_recall_fscore_support(y,pred,labels=np.arange(len(labels)),zero_division=0)
    return {'macro_f1':float(f1.mean()),'accuracy':float(accuracy_score(y,pred)),
      'precision_macro':float(precision.mean()),'recall_macro':float(recall.mean()),
      'loss':float(-np.log(np.maximum(p[np.arange(len(y)),y],1e-15)).mean()),
      'per_class':[{'label':l,'precision':float(precision[i]),'recall':float(recall[i]),'f1':float(f1[i]),'support':int(support[i])} for i,l in enumerate(labels)],
      'confusion_matrix':confusion_matrix(y,pred,labels=np.arange(len(labels))).tolist()}

def evaluate(model,loader,labels=None):
    model.eval();probs=[];truth=[]
    with torch.inference_mode():
        for batch in loader:
            truth.extend(batch['labels'].tolist())
            inputs={k:v.to(DEVICE) for k,v in batch.items() if k!='labels'}
            probs.extend(model(**inputs).logits.float().softmax(-1).cpu().tolist())
    p=np.asarray(probs)
    if not np.isfinite(p).all():raise RuntimeError('Probabilidades no finitas en evaluación')
    return p, metrics(truth,p,labels) if labels is not None else None


In [ ]:
# 12. Pesos de clase solo desde TRAIN, fórmula N/(K*n_clase)
train_relevant=[i for i,r in enumerate(train_rows) if r['label']!='no_relevante']
dev_relevant=[i for i,r in enumerate(dev_rows) if r['label']!='no_relevante']
counts_A=np.bincount([int(r['label']!='no_relevante') for r in train_rows],minlength=2)
counts_B=np.bincount([B_TO_ID[train_rows[i]['label']] for i in train_relevant],minlength=6)
assert (counts_A>0).all() and (counts_B>0).all()
weights_A=torch.tensor(len(train_rows)/(2*counts_A),dtype=torch.float32,device=DEVICE)
weights_B=torch.tensor(len(train_relevant)/(6*counts_B),dtype=torch.float32,device=DEVICE)
display(pd.DataFrame({'clase':A_LABELS,'n_train':counts_A,'peso':weights_A.cpu().tolist()}))
display(pd.DataFrame({'clase':B_LABELS,'n_train':counts_B,'peso':weights_B.cpu().tolist()}))


In [ ]:
# 13. AMP, early stopping y entrenamiento compartido
class EarlyStopping:
    def __init__(self):self.best=-math.inf;self.anchor=-math.inf;self.bad=0;self.best_epoch=None
    def update(self,score,epoch):
        if not math.isfinite(score):raise RuntimeError('Macro-F1 no finito')
        improved=score>self.best
        if improved:self.best=score;self.best_epoch=epoch
        if score>self.anchor+MIN_DELTA:self.anchor=score;self.bad=0
        else:self.bad+=1
        return improved,self.bad>=PATIENCE

def new_model(task,seed):
    seed_all(seed)  # A y B parten independientemente del mismo BETO base.
    labels=A_LABELS if task=='A' else B_LABELS
    model=AutoModelForSequenceClassification.from_pretrained(BASE_PATH,local_files_only=True,
      use_safetensors=False,num_labels=len(labels),classifier_dropout=DROPOUT,
      id2label=dict(enumerate(labels)),label2id={l:i for i,l in enumerate(labels)})
    assert model.dropout.p==DROPOUT and model.classifier.out_features==len(labels)
    model.gradient_checkpointing_enable()
    return model.to(DEVICE)

def optimizer_tools(model,loader):
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    total=MAX_EPOCHS*math.ceil(len(loader)/GRADIENT_ACCUMULATION_STEPS)
    warmup=math.floor(WARMUP_RATIO*total)
    def factor(step):
        if warmup and step<warmup:return step/warmup
        return max(0.,(total-step)/max(1,total-warmup))
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,factor)
    scaler=torch.amp.GradScaler('cuda')
    return optimizer,scheduler,scaler

def train_epoch(model,loader,weights,optimizer,scheduler,scaler,state,emit,description,max_attempts=None):
    model.train();optimizer.zero_grad(set_to_none=True)
    numerator_sum=denominator_sum=0.;nonfinite_losses=skips=updates=attempts=0
    iterator=iter(loader)
    progress=tqdm(total=math.ceil(len(loader)/GRADIENT_ACCUMULATION_STEPS),desc=description)
    try:
        while group:=list(islice(iterator,GRADIENT_ACCUMULATION_STEPS)):
            target=torch.cat([b['labels'] for b in group]).to(DEVICE)
            denominator=weights[target].sum()
            for batch in group:
                y=batch['labels'].to(DEVICE)
                inputs={k:v.to(DEVICE) for k,v in batch.items() if k!='labels'}
                with torch.autocast('cuda',dtype=torch.float16):logits=model(**inputs).logits
                numerator=torch.nn.functional.cross_entropy(logits.float(),y,weight=weights,reduction='sum')
                scaler.scale(numerator/denominator).backward()
                if bool(torch.isfinite(numerator)):
                    numerator_sum+=float(numerator.detach());denominator_sum+=float(weights[y].sum())
                else:nonfinite_losses+=1
            scaler.unscale_(optimizer)
            params=[p for p in model.parameters() if p.grad is not None]
            finite=all(bool(torch.isfinite(p.grad).all()) for p in params)
            norm=None
            if finite:
                norm=float(torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(p.grad.double()) for p in params])))
                coefficient=min(1.,GRAD_CLIP/(norm+1e-6))
                for p in params:p.grad.mul_(coefficient)
            scale_before=scaler.get_scale()
            scaler.step(optimizer);scaler.update()  # actuar ANTES de decidir persistencia
            skipped=scaler.get_scale()<scale_before
            optimizer.zero_grad(set_to_none=True)
            attempts+=1
            if skipped:skips+=1;state['consecutive_skips']+=1
            else:updates+=1;state['consecutive_skips']=0;scheduler.step()
            emit({'attempt':attempts,'skipped':skipped,'finite_gradients':finite,'gradient_norm':norm,
                  'scale_before':scale_before,'scale_after':scaler.get_scale(),'learning_rate':optimizer.param_groups[0]['lr']})
            progress.update(1)
            if state['consecutive_skips']>=MAX_CONSECUTIVE_AMP_SKIPS:
                raise RuntimeError(f'{description}: {MAX_CONSECUTIVE_AMP_SKIPS} omisiones AMP consecutivas tras recuperación de escala; abortado por persistencia.')
            if max_attempts is not None and attempts>=max_attempts:break
    finally:progress.close()
    return {'train_loss':numerator_sum/denominator_sum if denominator_sum else None,
            'updates':updates,'amp_skips':skips,'attempts':attempts,'nonfinite_loss_microbatches':nonfinite_losses,
            'learning_rate':optimizer.param_groups[0]['lr']}

def fit_task(task,seed):
    labels=A_LABELS if task=='A' else B_LABELS
    weights=weights_A if task=='A' else weights_B
    train_indices=list(range(len(train_rows))) if task=='A' else train_relevant
    dev_indices=list(range(len(dev_rows))) if task=='A' else dev_relevant
    loader=make_loader(train_rows,train_tokens,task,True,seed,train_indices)
    dev_loader=make_loader(dev_rows,dev_tokens,task,False,seed,dev_indices)
    destination=CHECKPOINTS/f'seed_{seed}'/f'model_{task}'
    if destination.exists():raise RuntimeError(f'Checkpoint ya existe: {destination}. No se sobrescribe una ejecución previa.')
    destination.mkdir(parents=True)
    model=None;optimizer=scheduler=scaler=None
    history=[];stopper=EarlyStopping();state={'consecutive_skips':0};start=time.monotonic()
    try:
        model=new_model(task,seed)
        optimizer,scheduler,scaler=optimizer_tools(model,loader)
        with (RESULTS/f'amp_seed_{seed}_model_{task}.jsonl').open('w',encoding='utf-8') as events:
            for epoch in range(1,MAX_EPOCHS+1):
                save_json(RESULTS/'execution_status.json',{'status':'running','seed':seed,'model':task,'epoch':epoch})
                def emit(event):
                    events.write(json.dumps({'seed':seed,'model':task,'epoch':epoch,**event},allow_nan=False)+'\n');events.flush()
                    if event['skipped']:tqdm.write(f'AMP omitido seed={seed} modelo={task} época={epoch}: escala {event["scale_before"]} → {event["scale_after"]}')
                train=train_epoch(model,loader,weights,optimizer,scheduler,scaler,state,emit,f'{seed}/{task} época {epoch}')
                probabilities,dev_metrics=evaluate(model,dev_loader,labels)
                improved,stop=stopper.update(dev_metrics['macro_f1'],epoch)
                history.append({'epoch':epoch,**train,'dev_loss':dev_metrics['loss'],'dev_macro_f1':dev_metrics['macro_f1'],
                                'dev_accuracy':dev_metrics['accuracy'],'dev_precision':dev_metrics['precision_macro'],
                                'dev_recall':dev_metrics['recall_macro']})
                pd.DataFrame(history).to_csv(RESULTS/f'history_seed_{seed}_model_{task}.csv',index=False)
                if improved:
                    model.save_pretrained(destination,safe_serialization=True);tokenizer.save_pretrained(destination)
                    save_json(destination/'selected.json',{'epoch':epoch,'metrics':dev_metrics,'labels':labels})
                    best_probs=probabilities.copy();best_metrics=dev_metrics
                print(seed,task,epoch,'train_loss=',train['train_loss'],'dev_macro_f1=',dev_metrics['macro_f1'])
                if stop:break
        if sum(h['updates'] for h in history)==0:raise RuntimeError('Ninguna actualización efectiva; entrenamiento inválido')
        del model;model=None
        del optimizer,scheduler,scaler;optimizer=scheduler=scaler=None
        release_gpu()
        model=AutoModelForSequenceClassification.from_pretrained(destination,local_files_only=True).to(DEVICE)
        check,_=evaluate(model,dev_loader,labels)
        if not np.allclose(check,best_probs,atol=1e-6,rtol=0):raise RuntimeError('Checkpoint recargado no reproduce probabilidades seleccionadas')
        # B también se evalúa en TODO DEV para combinar probabilidades, sin entrenar con NR.
        full_loader=make_loader(dev_rows,dev_tokens,task,False,seed)
        full_probs,_=evaluate(model,full_loader)
        info={'best_epoch':stopper.best_epoch,'metrics':best_metrics,'history':history,
              'seconds':time.monotonic()-start,'updates':sum(h['updates'] for h in history),
              'amp_skips':sum(h['amp_skips'] for h in history),'checkpoint_hashes':{p.name:digest(p) for p in destination.iterdir() if p.is_file()}}
        save_json(RESULTS/f'model_{task}_seed_{seed}.json',info)
        return full_probs,info
    finally:
        del model,optimizer,scheduler,scaler
        release_gpu()


In [ ]:
# 14. Entrenamiento MODELO A — DEV completo binario
def train_model_A(seed):return fit_task('A',seed)


In [ ]:
# 15. Entrenamiento MODELO B — solo TRAIN y DEV relevantes
def train_model_B(seed):return fit_task('B',seed)


In [ ]:
# 16. Soft gating: todas las filas pasan por A y B; no umbrales
def soft_gate(pa,pb):
    pa=np.asarray(pa,dtype=np.float64);pb=np.asarray(pb,dtype=np.float64)
    assert pa.ndim==pb.ndim==2 and pa.shape[1]==2 and pb.shape==(len(pa),6)
    assert np.isfinite(pa).all() and np.isfinite(pb).all() and (pa>=0).all() and (pb>=0).all()
    assert np.allclose(pa.sum(1),1,atol=1e-6) and np.allclose(pb.sum(1),1,atol=1e-6)
    final=np.zeros((len(pa),7),dtype=np.float64)
    final[:,NR_ID]=pa[:,0]
    for j,label in enumerate(B_LABELS):final[:,FINAL_TO_ID[label]]=pa[:,1]*pb[:,j]
    assert np.allclose(final.sum(1),1,atol=2e-6)
    return final

def error_counts(y,pred):
    y=np.asarray(y);pred=np.asarray(pred)
    result={'relevance_to_NR_errors':int(((y!=NR_ID)&(pred==NR_ID)).sum()),
            'NR_to_relevance_errors':int(((y==NR_ID)&(pred!=NR_ID)).sum()),
            'historical_internal_errors':int(((y!=NR_ID)&(pred!=NR_ID)&(y!=pred)).sum())}
    assert sum(result.values())==int((y!=pred).sum())
    return result


In [ ]:
# 17. Smoke test. True: solo prueba técnica; False: experimento completo en celdas 18–20.
RUN_SMOKE_TEST = True
results_by_seed={}
def run_smoke_test():
    probabilities={}
    for task in ['A','B']:
        indices=list(range(min(16,len(train_rows)))) if task=='A' else train_relevant[:16]
        loader=make_loader(train_rows,train_tokens,task,False,42,indices)
        model=None;optimizer=scheduler=scaler=None
        try:
            model=new_model(task,42);optimizer,scheduler,scaler=optimizer_tools(model,loader)
            events=[]
            # Hasta 8 intentos sobre la misma muestra técnica: permite recuperación AMP normal.
            total_updates=0;state={'consecutive_skips':0}
            for attempt in range(8):
                report=train_epoch(model,loader,weights_A if task=='A' else weights_B,
                    optimizer,scheduler,scaler,state,events.append,f'SMOKE {task}',max_attempts=1)
                total_updates+=report['updates']
                # Dos updates para comprobar avance del scheduler además del optimizer.
                if total_updates>=2:break
            assert total_updates>=2 and scheduler.last_epoch==total_updates,'Smoke: no se verificaron dos updates efectivos'
            test_loader=make_loader(train_rows,train_tokens,task,False,42,train_relevant[:4])
            probabilities[task],m=evaluate(model,test_loader,A_LABELS if task=='A' else B_LABELS)
            assert math.isfinite(m['loss'])
        finally:
            del model,optimizer,scheduler,scaler;release_gpu()
    p=soft_gate(probabilities['A'],probabilities['B'])
    assert p.shape==(4,7)
    print('SMOKE OK: forward, backward, updates, scheduler, evaluación y soft gating. Pesos descartados.')

protocol={'model':MODEL_NAME,'revision':MODEL_REVISION,'seeds':SEEDS,'max_length':MAX_LENGTH,
 'lr':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,'epochs':MAX_EPOCHS,'patience':PATIENCE,'min_delta':MIN_DELTA,
 'warmup_ratio':WARMUP_RATIO,'clip':GRAD_CLIP,'microbatch':MICRO_BATCH_SIZE,'accumulation':GRADIENT_ACCUMULATION_STEPS,
 'dropout':DROPOUT,'max_consecutive_amp_skips':MAX_CONSECUTIVE_AMP_SKIPS,'labels':FINAL_LABELS,
 'padding':'dynamic','input_file_hashes':input_hashes,'train_rows':len(train_rows),'dev_rows':len(dev_rows),
 'dev_identity_sha256':hashlib.sha256(json.dumps([(r['id'],text_hash(r),r['label']) for r in dev_rows],ensure_ascii=False).encode()).hexdigest(),
 'weights_A':weights_A.cpu().tolist(),'weights_B':weights_B.cpu().tolist(),'no_seed_selection':True}
if RUN_SMOKE_TEST:
    run_smoke_test()
else:
    if RESULTS.exists() or CHECKPOINTS.exists():
        raise RuntimeError('Ya existe /content/results o /content/checkpoints. Conserva esa ejecución; utiliza una sesión limpia, sin sobrescribir.')
    RESULTS.mkdir();CHECKPOINTS.mkdir()
    save_json(RESULTS/'protocol.json',protocol)
    save_json(RESULTS/'environment.json',{'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,
      'gpu':torch.cuda.get_device_name(),'platform':platform.platform(),
      'pip_freeze':subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)})

def run_seed(seed):
    if RUN_SMOKE_TEST:
        print(f'Seed {seed} omitida en modo smoke. Cambia RUN_SMOKE_TEST=False y ejecuta desde celda 17.');return
    if seed not in SEEDS or seed in results_by_seed:raise RuntimeError('Semilla desconocida o ya ejecutada')
    try:
        pa,info_a=train_model_A(seed)
        pb,info_b=train_model_B(seed)
        p=soft_gate(pa,pb)
        y=[FINAL_TO_ID[r['label']] for r in dev_rows];pred=p.argmax(1)
        result={'seed':seed,'metrics':metrics(y,p,FINAL_LABELS),'best_epoch_A':info_a['best_epoch'],
          'best_epoch_B':info_b['best_epoch'],'f1_A_relevance':info_a['metrics']['macro_f1'],
          'f1_B_historical':info_b['metrics']['macro_f1'],**error_counts(y,pred)}
        with (RESULTS/f'hierarchical_seed_{seed}_predictions.jsonl').open('w',encoding='utf-8') as f:
            for i,r in enumerate(dev_rows):
                item={'id':r['id'],'text_sha256':text_hash(r),'reference':r['label'],'prediction':FINAL_LABELS[pred[i]],
                      'probabilities':p[i].tolist(),'labels_order':FINAL_LABELS,'p_relevant':float(pa[i,1]),
                      'p_not_relevant':float(pa[i,0]),'seed':seed,
                      **{k:r[k] for k in ('family','component','source_id') if k in r}}
                f.write(json.dumps(item,ensure_ascii=False,allow_nan=False)+'\n')
        save_json(RESULTS/f'hierarchical_seed_{seed}_metrics.json',result)
        results_by_seed[seed]=result
        save_json(RESULTS/'execution_status.json',{'status':'seed_complete','seed':seed,'completed_seeds':list(results_by_seed)})
    except BaseException as exc:
        previous=json.loads((RESULTS/'execution_status.json').read_text()) if (RESULTS/'execution_status.json').exists() else {}
        save_json(RESULTS/'execution_status.json',{**previous,'status':'failed_or_interrupted','seed':seed,'error':repr(exc)})
        print(f'Interrumpido seed={seed}, último estado: {previous}. Resultados parciales conservados; no media incompleta.')
        raise


In [ ]:
# 18. Ejecución seed 42
run_seed(42)


In [ ]:
# 19. Ejecución seed 123
run_seed(123)


In [ ]:
# 20. Ejecución seed 2026
run_seed(2026)


In [ ]:
# 21. Métricas finales
complete=not RUN_SMOKE_TEST and set(results_by_seed)==set(SEEDS)
if not complete:
    print('Sin resumen científico: modo smoke o ejecución incompleta.')
else:
    rows=[];per_class=[]
    for seed in SEEDS:
        r=results_by_seed[seed]
        rows.append({'seed':seed,'macro_f1':r['metrics']['macro_f1'],'accuracy':r['metrics']['accuracy'],
          **{k:r[k] for k in ('best_epoch_A','best_epoch_B','f1_A_relevance','f1_B_historical',
                             'relevance_to_NR_errors','NR_to_relevance_errors','historical_internal_errors')}})
        per_class.extend({'seed':seed,**p} for p in r['metrics']['per_class'])
    metrics_table=pd.DataFrame(rows);per_class_table=pd.DataFrame(per_class)
    display(metrics_table);display(per_class_table)
    fig,axes=plt.subplots(1,2,figsize=(13,4))
    for ax,task in zip(axes,['A','B']):
        for seed in SEEDS:
            h=pd.read_csv(RESULTS/f'history_seed_{seed}_model_{task}.csv')
            ax.plot(h.epoch,h.dev_macro_f1,marker='o',label=f'seed {seed}')
        ax.set(title=f'Modelo {task}: DEV macro-F1',xlabel='Época',ylabel='Macro-F1');ax.legend();ax.grid(alpha=.25)
    fig.tight_layout();fig.savefig(RESULTS/'curvas_dev.png',dpi=160);plt.show()


In [ ]:
# 22. Matrices de confusión: filas reales, columnas predichas
if complete:
    fig,axes=plt.subplots(1,3,figsize=(17,5));short=['MIL','COL','IDE','LID','NR','REP','SOC']
    for ax,seed in zip(axes,SEEDS):
        cm=np.asarray(results_by_seed[seed]['metrics']['confusion_matrix'])
        ax.imshow(cm,cmap='Blues');ax.set(title=f'Seed {seed}',xlabel='Predicción',ylabel='Referencia')
        ax.set_xticks(range(7),short,rotation=45);ax.set_yticks(range(7),short)
        for i in range(7):
            for j in range(7):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
        pd.DataFrame(cm,index=FINAL_LABELS,columns=FINAL_LABELS).to_csv(RESULTS/f'hierarchical_confusion_seed_{seed}.csv')
    fig.tight_layout();fig.savefig(RESULTS/'confusiones.png',dpi=160);plt.show()


In [ ]:
# 23. Errores de la predicción FINAL soft-gated, no del argmax binario aislado
if complete:
    display(metrics_table[['seed','relevance_to_NR_errors','NR_to_relevance_errors','historical_internal_errors']])
    print('Las tres categorías son disjuntas y suman todos los errores. f1_A_relevance es macro-F1 binario; f1_B_historical es macro-F1 en DEV relevante.')


In [ ]:
# 24. Resumen mean ± std (muestral, ddof=1)
if complete:
    class_summary=per_class_table.groupby('label',sort=False).agg(f1_mean=('f1','mean'),f1_std=('f1','std')).reindex(FINAL_LABELS)
    summary={'seeds':SEEDS,'labels':FINAL_LABELS,'protocol':protocol,'std_ddof':1,
      'macro_f1_mean':float(metrics_table.macro_f1.mean()),'macro_f1_std':float(metrics_table.macro_f1.std(ddof=1)),
      'accuracy_mean':float(metrics_table.accuracy.mean()),'accuracy_std':float(metrics_table.accuracy.std(ddof=1)),
      'per_class':class_summary.reset_index().to_dict('records'),'results':[results_by_seed[s] for s in SEEDS],
      'interpretation':'DEV expuesto y usado para seleccionar A/B; no test independiente. Sin selección de semillas.'}
    print(f"Macro-F1: {summary['macro_f1_mean']:.6f} ± {summary['macro_f1_std']:.6f}")
    print(f"Accuracy: {summary['accuracy_mean']:.6f} ± {summary['accuracy_std']:.6f}")
    display(class_summary)
    fig,ax=plt.subplots(figsize=(10,4));x=np.arange(7)
    ax.bar(x,class_summary.f1_mean,yerr=class_summary.f1_std,capsize=4)
    ax.set_xticks(x,['MIL','COL','IDE','LID','NR','REP','SOC']);ax.set(ylabel='F1 medio ± std',ylim=(0,1.05))
    fig.tight_layout();fig.savefig(RESULTS/'f1_por_clase.png',dpi=160);plt.show()


In [ ]:
# 25. Comparación opcional: puedes subir el summary.json plano renombrado flat_beto_results.json
# Formato aceptado: {"results":[{"seed":42,"metrics":{"macro_f1":0.0,"accuracy":0.0}}, ...]}
# También acepta lista de filas {seed,macro_f1,accuracy}. Exactamente las tres semillas.
UPLOAD_FLAT_RESULTS = False
if complete:
    if UPLOAD_FLAT_RESULTS:
        optional=files.upload()
        flat_uploaded=optional.get('flat_beto_results.json')
    if flat_uploaded is None:
        print('Sin resultados planos: se continúa y se guarda la evaluación jerárquica.')
    else:
        flat=json.loads(flat_uploaded.decode('utf-8-sig'))
        flat_rows=flat if isinstance(flat,list) else flat.get('results')
        if not isinstance(flat_rows,list) or len(flat_rows)!=3:raise ValueError('Flat: se requieren las tres semillas, sin duplicados')
        converted=[]
        for r in flat_rows:
            m=r.get('metrics',r);converted.append({'seed':int(r['seed']),'flat_macro_f1':float(m['macro_f1']),'accuracy':float(m['accuracy'])})
        f=pd.DataFrame(converted)
        if set(f.seed)!=set(SEEDS) or f.seed.duplicated().any():raise ValueError('Flat: semillas distintas de 42/123/2026')
        if not np.isfinite(f[['flat_macro_f1','accuracy']].to_numpy()).all() or not f[['flat_macro_f1','accuracy']].ge(0).all().all() or not f[['flat_macro_f1','accuracy']].le(1).all().all():raise ValueError('Flat: métricas fuera de [0,1]')
        if isinstance(flat,dict) and 'labels' in flat and flat['labels']!=FINAL_LABELS:raise ValueError('Flat: orden de etiquetas diferente')
        if isinstance(flat,dict) and flat.get('protocol',{}).get('dev_identity_sha256') not in (None,protocol['dev_identity_sha256']):raise ValueError('Flat: identidad DEV diferente')
        print('Comparación descriptiva. Si el JSON plano no trae hashes, la identidad TRAIN/DEV no puede verificarse automáticamente; deben ser los mismos 758/160.')
        comparison=metrics_table[['seed','macro_f1']].rename(columns={'macro_f1':'hierarchical_macro_f1'}).merge(f[['seed','flat_macro_f1']],on='seed')
        comparison['delta']=comparison.hierarchical_macro_f1-comparison.flat_macro_f1
        display(comparison[['seed','flat_macro_f1','hierarchical_macro_f1','delta']])
        models=pd.DataFrame([{'modelo':'plano','macro_f1_mean':f.flat_macro_f1.mean(),'std':f.flat_macro_f1.std(ddof=1),'accuracy_mean':f.accuracy.mean()},
            {'modelo':'jerárquico','macro_f1_mean':summary['macro_f1_mean'],'std':summary['macro_f1_std'],'accuracy_mean':summary['accuracy_mean']}])
        display(models)
        comparison.to_csv(RESULTS/'comparison_by_seed.csv',index=False);models.to_csv(RESULTS/'comparison_models.csv',index=False)
        ax=comparison.set_index('seed')[['flat_macro_f1','hierarchical_macro_f1']].plot.bar(rot=0,ylabel='Macro-F1 DEV',figsize=(8,4))
        ax.figure.tight_layout();ax.figure.savefig(RESULTS/'comparison_by_seed.png',dpi=160);plt.show()


In [ ]:
# 26. Guardado de artefactos (predicciones y checkpoints ya se guardaron incrementalmente)
if complete:
    metrics_table.to_csv(RESULTS/'hierarchical_metrics_by_seed.csv',index=False)
    per_class_table.to_csv(RESULTS/'hierarchical_per_class.csv',index=False)
    save_json(RESULTS/'hierarchical_summary.json',summary)
    save_json(RESULTS/'execution_status.json',{'status':'complete','seeds':SEEDS,'model_trajectories':6})
    save_json(RESULTS/'artifact_hashes.json',{str(p.relative_to(RESULTS)):digest(p) for p in RESULTS.rglob('*') if p.is_file() and p.name!='artifact_hashes.json'})
    print('Resultados:',RESULTS,'Mejores checkpoints A/B de las tres semillas:',CHECKPOINTS)
else:print('Modo smoke/incompleto: no se genera un resumen científico.')


In [ ]:
# 27. Descargar results.zip; checkpoints opcionalmente (seis modelos, varios GB)
DOWNLOAD_CHECKPOINTS = False
if complete:
    archive=shutil.make_archive('/content/results','zip',root_dir=RESULTS)
    files.download(archive)
    if DOWNLOAD_CHECKPOINTS:
        for seed in SEEDS:
            archive=shutil.make_archive(f'/content/checkpoints_seed_{seed}','zip',root_dir=CHECKPOINTS/f'seed_{seed}')
            files.download(archive)
else:print('Smoke test terminado. Para entrenar cambia RUN_SMOKE_TEST=False y ejecuta desde celda 17.')
